# Manual Model vs. Pairwise Comparison Evaluation

For each pairwise comparison, compute what each person's manual scoring model would have predicted (A or B), and compare against their actual choice.

**Design decisions:**
- `total_donation` dropped (inconsistent scale between manual model and pairwise data)
- Poverty index 7 treated as 6 (`60+%`)
- Persons with no manual model skipped (D1, D5, R4, R6, V2)
- If a person's manual model is missing for a feature (was all-zero/degenerate), that feature contributes 0 to both sides — correct, since all-zero means no differentiation

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Index -> CSV column name mappings for each feature
INDEX_MAPS = {
    'size': {
        0: '<50', 1: '50-100', 2: '100-500', 3: '500-1000', 4: '1000'
    },
    'access': {
        0: 'normal', 1: 'low', 2: 'extremely low'
    },
    'income': {
        0: '0 - 20k', 1: '20k - 40k', 2: '40k - 60k',
        3: '60k - 80k', 4: '80k - 100k', 5: '> 100k'
    },
    'poverty': {
        0: '0 - 10%', 1: '10% - 20%', 2: '20% - 30%',
        3: '30% - 40%', 4: '40% - 50%', 5: '50% - 60%',
        6: '60% +', 7: '60% +'  # index 7 treated as 6
    },
    'last_donation': {
        0: 'never', **{i: str(i) for i in range(1, 13)}
    },
    'dist': {
        0: '15', 1: '30', 2: '45', 3: '60+'
    },
}

# CSV filename suffix for each feature
CSV_NAMES = {
    'size':          'organization_size',
    'access':        'access',
    'income':        'income',
    'poverty':       'poverty',
    'last_donation': 'last_donation',
    'dist':          'distance',
}

In [3]:
# Load manual scoring models: feature -> DataFrame (person x score_columns)
manual = {}
for feat, csv_name in CSV_NAMES.items():
    df = pd.read_csv(f'Manual scoring model database_sheet.xlsx - {csv_name}.csv')
    df = df.set_index('person')
    df = df.apply(pd.to_numeric, errors='coerce').fillna(0)
    manual[feat] = df
    print(f'{feat}: {len(df)} persons, columns: {list(df.columns)}')

size: 16 persons, columns: ['<50', '50-100', '100-500', '500-1000', '1000']
access: 15 persons, columns: ['extremely low', 'low', 'normal']
income: 13 persons, columns: ['> 100k', '0 - 20k', '20k - 40k', '40k - 60k', '60k - 80k', '80k - 100k']
poverty: 15 persons, columns: ['0 - 10%', '10% - 20%', '20% - 30%', '30% - 40%', '40% - 50%', '50% - 60%', '60% +']
last_donation: 14 persons, columns: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', 'never']
dist: 13 persons, columns: ['15', '30', '45', '60+']


In [4]:
def lookup_score(person, feature, index):
    """Return the manual model score for a person/feature/index combo.
    Returns 0 if person has no model for this feature (was degenerate)."""
    df = manual[feature]
    idx_map = INDEX_MAPS[feature]
    if person not in df.index:
        return 0.0
    col = idx_map[index]
    return float(df.loc[person, col])

In [5]:
# Load pairwise comparison data
pw = pd.read_csv('food_rescue_combined copy.csv')

# Persons who have at least one manual model entry (use size as proxy — everyone has it)
persons_with_model = set(manual['size'].index)
print('Persons with manual model:', sorted(persons_with_model))
print('Persons in pairwise data: ', sorted(pw['personID'].unique()))
skipped = set(pw['personID'].unique()) - persons_with_model
print('Skipped (no model):       ', sorted(skipped))

Persons with manual model: ['D2', 'D4', 'F1', 'F2', 'F3', 'R1', 'R2', 'R3', 'R5', 'R7', 'R8', 'V1', 'V3', 'V5', 'V6', 'V7']
Persons in pairwise data:  ['D1', 'D2', 'D5', 'F1', 'F2', 'R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'V1', 'V2', 'V3', 'V5', 'V6', 'V7']
Skipped (no model):        ['D1', 'D5', 'R4', 'R6', 'V2']


In [6]:
FEATURES = list(CSV_NAMES.keys())

rows = []
for _, row in pw.iterrows():
    person = row['personID']
    if person not in persons_with_model:
        continue

    score_a, score_b = 0.0, 0.0
    feat_scores = {}
    for feat in FEATURES:
        idx_a = int(row[f'{feat}_A'])
        idx_b = int(row[f'{feat}_B'])
        sa = lookup_score(person, feat, idx_a)
        sb = lookup_score(person, feat, idx_b)
        score_a += sa
        score_b += sb
        feat_scores[f'score_{feat}_A'] = sa
        feat_scores[f'score_{feat}_B'] = sb

    if score_a > score_b:
        prediction = 'A'
    elif score_b > score_a:
        prediction = 'B'
    else:
        prediction = 'tie'

    actual = row['AorB']
    match = None if prediction == 'tie' else (prediction == actual)

    rows.append({
        **row.to_dict(),
        **feat_scores,
        'model_score_A': score_a,
        'model_score_B': score_b,
        'model_prediction': prediction,
        'matches_actual': match,
    })

results = pd.DataFrame(rows)
print(f'Total rows:    {len(results)}')
print(f'Ties:          {(results["model_prediction"] == "tie").sum()}')
print(f'Non-tie rows:  {(results["model_prediction"] != "tie").sum()}')

Total rows:    631
Ties:          25
Non-tie rows:  606


In [7]:
# Per-person accuracy (excluding ties)
non_tie = results[results['model_prediction'] != 'tie'].copy()
per_person = non_tie.groupby('personID').agg(
    n_comparisons=('matches_actual', 'count'),
    n_correct=('matches_actual', 'sum'),
).assign(accuracy=lambda d: d['n_correct'] / d['n_comparisons'])
tie_counts = results[results['model_prediction'] == 'tie'].groupby('personID').size().rename('n_ties')
per_person = per_person.join(tie_counts, how='left').fillna({'n_ties': 0})
per_person['n_ties'] = per_person['n_ties'].astype(int)
print(per_person.sort_values('accuracy', ascending=False).to_string())

          n_comparisons n_correct  accuracy  n_ties
personID                                           
V5                   43        36  0.837209       3
V7                   43        36  0.837209       2
R3                   46        38  0.826087       1
V3                   45        37  0.822222       0
R5                   45        35  0.777778       0
R1                   37        27   0.72973       3
R2                   39        27  0.692308       6
D2                   45        31  0.688889       0
F2                   41        28  0.682927       4
R8                   46        31  0.673913       1
R7                   46        30  0.652174       1
F1                   45        29  0.644444       0
V6                   45        29  0.644444       0
V1                   40        24       0.6       4


In [8]:
# Overall accuracy
overall = non_tie['matches_actual'].mean()
print(f'Overall accuracy (excluding ties): {overall:.3f}  '
      f'({int(non_tie["matches_actual"].sum())} / {len(non_tie)})')

Overall accuracy (excluding ties): 0.723  (438 / 606)


In [9]:
# Save full results
results.to_csv('pairwise_model_predictions.csv', index=False)
print('Saved pairwise_model_predictions.csv')

Saved pairwise_model_predictions.csv


## Ties and Mismatches Per Person

In [10]:
show = ['num_Q', 'food',
        'size_A', 'access_A', 'income_A', 'poverty_A', 'last_donation_A', 'dist_A',
        'size_B', 'access_B', 'income_B', 'poverty_B', 'last_donation_B', 'dist_B',
        'AorB', 'model_prediction', 'model_score_A', 'model_score_B']

all_ties = []
all_mismatches = []

for person in sorted(results['personID'].unique()):
    p = results[results['personID'] == person]
    ties = p[p['model_prediction'] == 'tie'][['personID'] + show].copy()
    mismatches = p[p['matches_actual'] == False][['personID'] + show].copy()
    all_ties.append(ties)
    all_mismatches.append(mismatches)

    print(f'\n===== {person} =====')
    print(f'TIES ({len(ties)}):')
    display(ties[show].reset_index(drop=True)) if len(ties) else print('  none')
    print(f'MISMATCHES ({len(mismatches)}):')
    display(mismatches[show].reset_index(drop=True)) if len(mismatches) else print('  none')


===== D2 =====
TIES (0):
  none
MISMATCHES (14):


,num_Q,food,size_A,access_A,income_A,poverty_A,last_donation_A,dist_A,size_B,access_B,income_B,poverty_B,last_donation_B,dist_B,AorB,model_prediction,model_score_A,model_score_B
0,1,1,4,0,5,0,6,2,4,0,0,6,2,3,A,B,15.0,30.0
1,2,0,0,2,5,1,9,2,4,0,5,0,12,0,B,A,35.0,19.0
2,6,0,0,2,3,0,11,0,4,2,3,1,0,2,B,A,37.0,34.0
3,12,0,2,0,5,0,7,2,1,0,5,1,7,1,A,B,20.0,22.0
4,16,1,3,0,1,3,4,2,2,2,5,0,1,3,B,A,30.0,24.0
5,23,0,4,2,3,0,8,1,0,1,1,4,3,1,A,B,34.0,40.0
6,25,1,4,1,2,2,4,0,3,2,5,1,8,1,A,B,31.0,33.0
7,28,0,4,0,0,7,7,1,2,1,1,2,7,0,A,B,37.0,40.0
8,30,0,4,2,5,0,0,2,4,0,1,3,9,0,A,B,32.0,33.0
9,34,1,3,0,4,1,10,0,4,2,3,0,8,2,A,B,20.0,34.0



===== F1 =====
TIES (0):
  none
MISMATCHES (16):


,num_Q,food,size_A,access_A,income_A,poverty_A,last_donation_A,dist_A,size_B,access_B,income_B,poverty_B,last_donation_B,dist_B,AorB,model_prediction,model_score_A,model_score_B
0,3,1,3,1,3,1,5,3,3,2,1,0,5,1,A,B,38.0,48.0
1,6,1,1,0,4,1,10,1,4,0,1,2,4,1,A,B,42.0,50.0
2,8,1,4,0,5,0,7,2,2,2,2,1,2,3,A,B,22.0,45.0
3,13,1,2,1,0,0,4,3,4,0,2,1,10,1,B,A,49.0,46.0
4,14,1,0,1,3,1,11,0,1,1,2,3,3,0,A,B,53.0,58.0
5,16,0,4,0,3,1,10,0,1,2,4,1,12,2,A,B,42.0,48.0
6,18,1,0,2,5,1,0,2,3,2,1,2,2,1,A,B,46.0,55.0
7,19,1,2,1,4,1,6,0,1,2,3,1,7,2,A,B,39.0,48.0
8,21,1,0,0,0,2,12,3,3,2,5,1,12,2,B,A,69.0,39.0
9,22,1,0,1,3,1,4,3,0,2,3,0,3,1,B,A,43.0,42.0



===== F2 =====
TIES (4):


...

,num_Q,food,size_A,access_A,income_A,poverty_A,last_donation_A,dist_A,size_B,access_B,income_B,poverty_B,last_donation_B,dist_B,AorB,model_prediction,model_score_A,model_score_B
0,4,1,2,2,0,6,6,1,3,2,5,1,2,1,B,A,55.0,44.0
1,8,1,3,0,1,4,4,2,2,0,1,1,7,0,A,B,32.0,55.0
2,11,1,3,2,5,0,2,3,4,2,3,0,2,3,A,B,29.0,32.0
3,12,0,3,1,2,2,6,1,0,2,4,1,0,0,A,B,48.0,65.0
4,14,0,3,1,1,3,7,2,4,0,3,1,5,3,B,A,41.0,33.0
5,16,1,2,2,2,3,10,2,2,1,4,0,12,0,A,B,50.0,65.0
6,19,0,3,2,5,1,0,0,0,0,0,0,1,2,B,A,67.0,27.0
7,20,1,4,2,4,1,7,0,1,2,3,1,0,0,A,B,63.0,65.0
8,23,1,0,2,3,0,6,0,3,2,4,0,8,3,B,A,58.0,42.0
9,26,0,3,0,0,5,9,3,3,2,0,3,8,2,A,B,42.0,50.0



===== V7 =====
TIES (2):


,num_Q,food,size_A,access_A,income_A,poverty_A,last_donation_A,dist_A,size_B,access_B,income_B,poverty_B,last_donation_B,dist_B,AorB,model_prediction,model_score_A,model_score_B
0,6,1,0,0,1,2,2,0,3,1,5,0,0,1,B,tie,9.0,9.0
1,34,1,3,2,2,1,8,0,4,1,0,1,5,2,B,tie,13.5,13.5


MISMATCHES (7):


,num_Q,food,size_A,access_A,income_A,poverty_A,last_donation_A,dist_A,size_B,access_B,income_B,poverty_B,last_donation_B,dist_B,AorB,model_prediction,model_score_A,model_score_B
0,7,0,3,1,3,1,7,1,4,0,2,0,5,2,B,A,11.5,8.0
1,9,0,2,2,1,0,1,0,3,2,3,1,6,1,B,A,12.0,11.5
2,13,0,1,0,1,1,2,0,0,0,3,0,0,1,B,A,8.5,7.0
3,20,0,4,2,2,2,12,1,3,2,2,3,12,0,A,B,14.0,16.0
4,21,1,0,2,2,0,10,2,3,1,1,1,4,0,A,B,12.0,12.5
5,31,0,1,1,1,3,7,3,4,1,2,2,0,0,B,A,15.0,13.0
6,43,1,3,1,2,2,9,1,1,0,3,0,0,1,B,A,13.0,7.0


In [11]:
pd.concat(all_ties).to_csv('ties.csv', index=False)
pd.concat(all_mismatches).to_csv('mismatches.csv', index=False)
print('Saved ties.csv and mismatches.csv')

Saved ties.csv and mismatches.csv


## Conflicted Decisions with Full Feature Data

In [ ]:
difficulty = pd.read_csv('transcript_difficulty.csv')

# Join on personID + num_Q
merged = difficulty.merge(results, on=['personID', 'num_Q'], how='left')

# Focus on conflicted decisions that have pairwise data
conflicted = merged[merged['conflicted'] & merged['AorB'].notna()].copy()

feature_cols = ['size_A','access_A','income_A','poverty_A','last_donation_A','dist_A',
                'size_B','access_B','income_B','poverty_B','last_donation_B','dist_B']
show_cols = ['personID','num_Q','difficulty_score','evidence',
             'AorB','model_prediction','matches_actual'] + feature_cols

print(f'Conflicted decisions with pairwise features: {len(conflicted)}')
display(conflicted[show_cols].reset_index(drop=True))

In [ ]:
# Save merged conflicted decisions
conflicted[show_cols].to_csv('conflicted_with_features.csv', index=False)
print('Saved conflicted_with_features.csv')